# M3L2 E02 - RAG mini: el pipeline completo (Resolution)

## Este notebook necesita API key de OpenAI y FAISS


In [ ]:
# !pip install faiss-cpu langchain langchain-openai langchain-community

import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")


## Mapa de conceptos

| Concepto | Rol | Componente LangChain |
|---|---|---|
| Embeddings | Convierte texto en vectores | `OpenAIEmbeddings()` |
| Vector Store | Almacena y busca por similitud | `FAISS.from_texts()` |
| Retriever | Interfaz de busqueda | `vectorstore.as_retriever()` |
| PromptTemplate | Estructura el prompt | `ChatPromptTemplate.from_messages()` |
| LLM | Genera la respuesta | `ChatOpenAI()` |
| LCEL | Conecta todo | `{...} \| prompt \| llm \| parser` |


## Bloque 1 - Script legacy


In [ ]:
from openai import OpenAI

client = OpenAI()

DOCS_EMPRESA = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]


def answer_script_legacy(question: str) -> str:
    context = "\n".join(DOCS_EMPRESA)  # manda TODO el contexto siempre
    prompt_text = (
        "Eres un asistente de RRHH. Responde usando solo el contexto dado.\n\n"
        f"Contexto:\n{context}\n\nPregunta:\n{question}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0,
    )
    return response.choices[0].message.content


print("Script legacy:")
print(answer_script_legacy("Cuantos dias de vacaciones tengo?"))


## Bloque 2 - Pipeline RAG modular con LangChain


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

TEXTOS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
parser = StrOutputParser()

print(f"LLM: {llm.model_name}")


In [ ]:
# TODO 1: crear el vector store
vectorstore = FAISS.from_texts(TEXTOS, embeddings)
print(f"Vector store: {type(vectorstore).__name__}")


In [ ]:
# TODO 2: crear el retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Retriever: {type(retriever).__name__}")

# Probar el retriever directamente
docs_test = retriever.invoke("vacaciones")
print(f"Documentos recuperados: {len(docs_test)}")
for i, doc in enumerate(docs_test):
    print(f"  Doc {i+1}: {doc.page_content}")


In [ ]:
# TODO 3: crear el prompt RAG
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente de RRHH. "
        "Responde usando solo el contexto proporcionado. "
        "Si la respuesta no esta en el contexto, indica que no tienes informacion suficiente."
    ),
    (
        "human",
        "Contexto:\n{context}\n\nPregunta:\n{question}"
    )
])
print(f"Prompt RAG: {rag_prompt.input_variables}")


In [ ]:
def format_docs(docs) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


# TODO 4: componer la RAG chain con LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser
)

print(f"RAG chain: {type(rag_chain).__name__}")


In [ ]:
# TODO 5: invocar la RAG chain
respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
print(f"Respuesta RAG: {respuesta}")


## Bloque 3 - Debugging modular


In [ ]:
question = "Cuantos dias de vacaciones tengo?"

print("=== Inspeccion del pipeline ===")

print("[Paso 1] Documentos recuperados:")
docs = retriever.invoke(question)
for i, doc in enumerate(docs):
    print(f"  Doc {i+1}: {doc.page_content}")
print()

print("[Paso 2] Contexto formateado:")
context = format_docs(docs)
print(context)
print()

print("[Paso 3] Prompt final:")
messages = rag_prompt.format_messages(context=context, question=question)
for msg in messages:
    print(f"  [{msg.type.upper()}] {msg.content[:80]}...")
print()

print("Cada componente es inspeccionable por separado.")
print("Si la respuesta es mala, sabemos en que paso buscar el problema.")


## Checks


In [ ]:
def run_checks():
    assert vectorstore is not None
    assert retriever is not None
    assert rag_prompt is not None
    assert rag_chain is not None

    docs = retriever.invoke("vacaciones")
    assert len(docs) > 0
    assert len(docs) <= 2
    contenidos = [doc.page_content for doc in docs]
    assert any("vacaciones" in c.lower() or "15 dias" in c.lower() for c in contenidos)

    respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
    assert isinstance(respuesta, str)
    assert len(respuesta) > 0
    assert "15" in respuesta

    docs_remoto = retriever.invoke("trabajo remoto")
    contenidos_remoto = [doc.page_content for doc in docs_remoto]
    assert any("remoto" in c.lower() for c in contenidos_remoto)

    print("M3L2 E02 Resolution checks passed")


run_checks()


## Cierre

| Aspecto | Script legacy | Pipeline RAG con LangChain |
|---|---|---|
| Retrieval | Manda todos los docs siempre | Solo los k mas relevantes |
| Ingestion | Se repite en cada consulta | Se hace una vez, se persiste |
| Prompt | F-string hardcodeado | ChatPromptTemplate reutilizable |
| Modelo | Hardcodeado en la funcion | Objeto reemplazable |
| Debugging | Opaco | Cada paso es inspeccionable |
